In [1]:
import pandas as pd
import time
from openai import OpenAI

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

In [8]:
# ---- 1. Load the disagreement file from Day 6 + full merged data ----
rule_right_llm_wrong = pd.read_csv(r"C:\Users\Praphulla\Downloads\Research\data\processed\disagree_rule_right.csv")  # 67 cases
rule_preds = pd.read_csv(r"C:\Users\Praphulla\Downloads\Research\data\processed\abt_buy_rulebased_preds.csv")
llm_preds = pd.read_csv(r"C:\Users\Praphulla\Downloads\Research\data\processed\abt_buy_llm_zeroshot_preds.csv")

merged = rule_preds[['id_abt', 'id_buy', 'label', 'pred_label']].merge(
    llm_preds[['id_abt', 'id_buy', 'name_abt', 'name_buy', 'pred_label']],
    on=['id_abt', 'id_buy'], suffixes=('_rule', '_llm')
)

# Get the "both_correct" cases for our control sample
both_correct = merged[(merged['pred_label_rule'] == merged['label']) &
                       (merged['pred_label_llm'] == merged['label'])]
control_sample = both_correct.sample(50, random_state=1)

In [9]:
# ---- 2. Improved few-shot prompt with explicit SKU guidance ----
def classify_pair_v2(name_a, name_b, max_retries=3):
    prompt = f"""You are comparing two product listings to determine if they refer to the same product.

Pay close attention to shared model numbers or SKU codes (alphanumeric codes like '2349B001'). An exact code match is strong evidence of the same product, even if descriptive wording differs significantly.

Example: Product A: "Canon Deluxe Grey Leather Case - 2349B001". Product B: "Canon PSC-1000 Semi-Hard Leather Case - 2349B001". Answer: MATCH (same model code 2349B001).

Example: Product A: "Sony Black Headphones - MDR200". Product B: "Panasonic Blue Speaker - RQ500". Answer: NO_MATCH (different brands, different codes).

Now classify this pair:
Product A: {name_a}
Product B: {name_b}

Respond with only one word: MATCH or NO_MATCH."""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=5
            )
            return response.choices[0].message.content.strip(), None
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            return None, str(e)

def parse_response(raw_response):
    if raw_response is None:
        return None
    r = raw_response.upper()
    if r == "MATCH":
        return 1
    elif r == "NO_MATCH":
        return 0
    elif "NO_MATCH" in r:
        return 0
    elif "MATCH" in r:
        return 1
    return None


In [10]:
# ---- 3. Test on the 67 previously-wrong pairs ----
flip_results = []
for _, row in rule_right_llm_wrong.iterrows():
    raw, err = classify_pair_v2(row['name_abt'], row['name_buy'])
    pred = parse_response(raw)
    flip_results.append({
        'name_abt': row['name_abt'], 'name_buy': row['name_buy'],
        'label': row['label'], 'old_pred_llm': row['pred_label_llm'],
        'new_pred_llm': pred, 'raw_response': raw
    })

flip_df = pd.DataFrame(flip_results)
n_flipped_to_correct = ((flip_df['new_pred_llm'] == flip_df['label']) &
                         (flip_df['old_pred_llm'] != flip_df['label'])).sum()
print(f"Of {len(flip_df)} previously-wrong cases: {n_flipped_to_correct} now correct")

Of 67 previously-wrong cases: 28 now correct


In [11]:
# ---- 4. Test on the 50 control (previously correct) cases ----
control_results = []
for _, row in control_sample.iterrows():
    raw, err = classify_pair_v2(row['name_abt'], row['name_buy'])
    pred = parse_response(raw)
    control_results.append({
        'name_abt': row['name_abt'], 'name_buy': row['name_buy'],
        'label': row['label'], 'old_pred_llm': row['pred_label_llm'],
        'new_pred_llm': pred, 'raw_response': raw
    })

control_df = pd.DataFrame(control_results)
n_broken = ((control_df['new_pred_llm'] != control_df['label']) &
            (control_df['old_pred_llm'] == control_df['label'])).sum()
print(f"Of {len(control_df)} previously-correct cases: {n_broken} now broken")


Of 50 previously-correct cases: 1 now broken


In [13]:
# ---- 5. Save both for inspection ----
flip_df.to_csv('../data/processed/day7_flip_test.csv', index=False)
control_df.to_csv('../data/processed/day7_control_test.csv', index=False)

print(f"\nSummary: {n_flipped_to_correct}/{len(flip_df)} fixed, {n_broken}/{len(control_df)} broken")
print("Verdict: worth full re-run tomorrow if fixed >> broken")




Summary: 28/67 fixed, 1/50 broken
Verdict: worth full re-run tomorrow if fixed >> broken


In [23]:
control_df

,name_abt,name_buy,label,old_pred_llm,new_pred_llm,raw_response
0,Toshiba Black 1080p Upconversion DVD Recorder/...,Toshiba D-VR610 DVD VCR Combo,1,1,1,MATCH
1,Logitech 2.1 Multimedia Silver Speaker System ...,Logitech Z-2300 Multimedia Speaker System - 97...,1,1,1,MATCH
2,Apple 8GB Black 4th Generation iPod Nano - MB7...,Apple 8GB iPod nano Black (4th Generation) - M...,1,1,1,MATCH
3,Panasonic 5.8 GHz Black Expandable Digital Cor...,LaCie Little Disk Hard Drive - 301829,0,0,0,NO_MATCH
4,Yamaha High Performance Subwoofer In Black - Y...,Yamaha YST-FSW150 Subwoofer - YST-FSW150BL,1,1,1,MATCH
5,Samsung 19' Black Flat Panel Series 6 LCD HDTV...,Samsung 6 Series LN19A650 19' LCD TV,1,1,1,MATCH
6,Whirlpool Duet Sport 27' White Electric Dryer ...,Whirlpool Duet Sport 27'' Electric Dryer - WED...,1,1,1,MATCH
7,LG Washer Dryer White Stacking Kit - WSTK1,LG Electronics LG WSTK1 Washer/Dryer Stacking ...,1,1,1,MATCH
8,Canon Black Ink Cartridge - PG50,Canon PG-50 High Capacity Black Ink Cartridge ...,1,1,1,MATCH
9,Yamaha High Performance Subwoofer - Black Fin...,Yamaha YSTFSW100 Subwoofer - YST-FSW100BL,1,1,1,MATCH
